In [1]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

# Try import astropy LombScargle
try:
    from astropy.timeseries import LombScargle
except Exception:
    raise ImportError("Install astropy: pip install astropy")

In [ ]:
# === PATH CONFIGURATION ===
base_dir = r"/home/vitor_guez/Projeto_Thaisa/ztffp_final_data_2025"
output_dir = r"/home/vitor_guez/Projeto_Thaisa/power spectra/LC_standardized"
stats_dir = r"/home/vitor_guez/Projeto_Thaisa/power spectra/LC_statistics"
plot_dir = r"/home/vitor_guez/Projeto_Thaisa/power spectra/LC_standardized_plots"

# The ROOT_DIR used in the algorithm:
ROOT_DIR = base_dir

# The main output folder for this script:
SAVE_FOLDER = stats_dir   

# ensure all exist
os.makedirs(output_dir, exist_ok=True)
os.makedirs(stats_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(SAVE_FOLDER, exist_ok=True)


In [ ]:
# USER PARAMETERS (adjustable)
time_unit_days = True           # True if time columns are in days (MJD-like). If in seconds set False.
min_points_per_lc = 5           # require at least this many points in trimmed interval
n_freq_points_per_lc = 1000     # sampling for autopower (passed to autopower via samples_per_peak-like control)
time_round_days = 1.0           # round start/end to nearest this many days to find the "most common" interval; adjust
freq_round_decimals = 6         # ROUND decimals for matching identical frequencies (adjustable). Example: 6 keeps ~1e-6 precision
agg_method = "median"           # "median" or "mean" for aggregated PSD value
estimate_psd_uncertainty = True # use perturbation (y +/- dy) to estimate per-PSD uncertainties when flux_err exists
hist_bin_width = 1e-6           # bin width (Hz) for histogram of counts vs frequency

In [ ]:
# Helpers: robust read and column detection
def try_read_table(path):
    """Read csv/txt/fits into pandas DataFrame (fallbacks)."""
    ext = os.path.splitext(path)[1].lower()
    if ext in (".csv", ".txt", ".dat"):
        for sep in [",", r"\s+", "\t"]:
            try:
                df = pd.read_csv(path, sep=sep, engine="python")
                if df.shape[1] > 0:
                    return df
            except Exception:
                continue
        return pd.read_csv(path)
    elif ext in (".fits", ".fz"):
        from astropy.table import Table
        t = Table.read(path)
        return t.to_pandas()
    else:
        return pd.read_csv(path)

def detect_columns(df):
    """Return (time_col, flux_col, flux_err_col_or_None)."""
    cols_lower = [c.lower() for c in df.columns]
    time_keys = ['time','mjd','jd','date','t']
    flux_keys = ['flux','f','flux_obs','fluxo','mag','flux_density','fluxcal','flux_value']
    ferr_keys = ['flux_err','err','error','f_err','fluxerror','e_flux','sigma','fluxerr','_err']

    tcol = None; fcol = None; ecol = None
    for i,c in enumerate(cols_lower):
        if tcol is None and any(c == k for k in time_keys):
            tcol = df.columns[i]
        if fcol is None and any(k in c for k in flux_keys):
            fcol = df.columns[i]
        if ecol is None and any(k in c for k in ferr_keys):
            ecol = df.columns[i]

    # fallback heuristics
    if tcol is None:
        for i,c in enumerate(cols_lower):
            if 'time' in c or 'mjd' in c or 'jd' in c:
                tcol = df.columns[i]; break
    if fcol is None:
        numeric_cols = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
        for col in numeric_cols:
            if tcol is not None and col == tcol:
                continue
            fcol = col; break

    if fcol is None:
        raise ValueError("Flux column not found. Available columns: " + ", ".join(df.columns.astype(str)))
    return tcol, fcol, ecol

In [ ]:
# Find object subfolders under ROOT_DIR
obj_dirs = [d for d in sorted(glob.glob(os.path.join(ROOT_DIR, "*"))) if os.path.isdir(d)]
print(f"Found {len(obj_dirs)} object subdirectories under {ROOT_DIR}")

# For each LC file under each object folder, detect start and end times (in days)
start_list = []
end_list = []
file_records = []  # (obj_dir, file_path, start, end)

for obj in obj_dirs:
    files = sorted(glob.glob(os.path.join(obj, "*.*")))
    files = [f for f in files if os.path.isfile(f) and os.path.splitext(f)[1].lower() in ('.csv','.txt','.dat','.fits','.fz')]
    if len(files) == 0:
        continue
    for fp in files:
        try:
            df = try_read_table(fp)
        except Exception as e:
            # skip unreadable files
            continue
        try:
            tcol, fcol, ecol = detect_columns(df)
        except Exception:
            continue
        t = np.asarray(df[tcol], dtype=float)
        t = t[np.isfinite(t)]
        if len(t) < 2:
            continue
        # keep time in days for uniformity (assume input in days if time_unit_days True)
        if time_unit_days:
            t_days = t
        else:
            # convert seconds to days for start/end detection
            t_days = t / 86400.0
        start_list.append(t_days.min())
        end_list.append(t_days.max())
        file_records.append((obj, fp, t_days.min(), t_days.max()))

# If none found
if len(start_list) == 0:
    raise RuntimeError("No light curve files found or no readable time columns under ROOT_DIR.")

# Round starts/ends to determine the most common interval
# Use time_round_days (e.g. 1.0 day) as rounding granularity. This is adjustable.
starts_rounded = np.round(np.array(start_list) / time_round_days) * time_round_days
ends_rounded   = np.round(np.array(end_list)   / time_round_days) * time_round_days

# Build pairs and find the most common (mode) pair
pairs = list(zip(starts_rounded, ends_rounded))
# Count frequencies
from collections import Counter
pair_counts = Counter(pairs)
most_common_pair, most_common_count = pair_counts.most_common(1)[0]
common_start_days, common_end_days = most_common_pair
print(f"Most common (rounded) interval: start={common_start_days} d, end={common_end_days} d (count={most_common_count})")
print("Note: rounding granularity (days) used:", time_round_days)

# Now select files whose raw (unrounded) interval contains the exact common interval
# i.e., raw_start <= common_start AND raw_end >= common_end
selected_files = []
selected_objs = []
selected_coords = []  # tuple (obj_folder_name, ra, dec) parsed from folder name if possible

for (obj, fp, raw_start, raw_end) in file_records:
    # raw_start/raw_end are in days (we used that convention above)
    if raw_start <= common_start_days + 1e-12 and raw_end >= common_end_days - 1e-12:
        selected_files.append(fp)
        selected_objs.append(obj)
        # try to parse coordinates from obj folder name like "RA_DEC"
        base = os.path.basename(obj)
        ra_dec = (None, None)
        if "_" in base:
            parts = base.split("_")
            try:
                ra = float(parts[0]); dec = float(parts[1])
                ra_dec = (ra, dec)
            except Exception:
                ra_dec = (None, None)
        selected_coords.append((base, fp, ra_dec[0], ra_dec[1]))

print(f"Selected {len(selected_files)} files whose intervals contain the common interval.")

# Save selected files list and coords (high precision)
sel_df = pd.DataFrame(selected_coords, columns=["obj_folder", "file_path", "ra", "dec"])
sel_df.to_csv(os.path.join(SAVE_FOLDER, "selected_files_list.csv"), index=False, float_format="%.10f")
print("Saved selected_files_list.csv with coordinates (if parsed).")

# If no selected files, stop
if len(selected_files) == 0:
    raise RuntimeError("No files contained the common interval. Try reducing time_round_days or inspect data.")

In [ ]:
# For each selected file: read, trim to [common_start, common_end], compute LSD autopower PSD
# We'll collect per-file PSD freq arrays, power arrays, power_err arrays, and contributor labels
all_freqs = []
all_pows = []
all_powerrs = []
contributors = []   # matching length arrays of contributor labels (filename)

# We'll also keep a list of selected unique objects (folders) to save coordinates later
selected_obj_folders = sorted(list(set(selected_objs)))

for fp in tqdm(selected_files, desc="Processing selected files (trim + PSD)"):
    try:
        df = try_read_table(fp)
    except Exception:
        continue
    try:
        tcol, fcol, ecol = detect_columns(df)
    except Exception:
        continue

    t = np.asarray(df[tcol], dtype=float)
    y = np.asarray(df[fcol], dtype=float)
    dy = None
    if ecol is not None and ecol in df.columns:
        try:
            dy = np.asarray(df[ecol], dtype=float)
        except Exception:
            dy = None

    # convert to days if not already (we used that for detection earlier)
    if not time_unit_days:
        t_days = t / 86400.0
    else:
        t_days = t

    # trim to common interval (in days)
    mask = (t_days >= common_start_days - 1e-12) & (t_days <= common_end_days + 1e-12)
    if mask.sum() < min_points_per_lc:
        continue
    t_trim = t_days[mask]
    y_trim = y[mask]
    dy_trim = dy[mask] if (dy is not None) else None

    # convert trimmed time to seconds relative to its own start for PSD calculation
    t_sec = (t_trim - t_trim.min()) * 86400.0

    # define PSD frequency bounds for this trimmed segment
    duration = t_sec.max() - t_sec.min()
    if duration <= 0:
        continue
    fmin = 1.0 / duration
    dt_min = np.min(np.diff(np.sort(t_sec)))
    if dt_min <= 0:
        continue
    fmax = 0.5 / dt_min

    # compute autopower via LombScargle.autopower
    # parameters: use samples_per_peak ~ 5 for good resolution; autopower returns freq, power
    try:
        ls = LombScargle(t_sec, y_trim, dy_trim)
        freq_obj, power_obj = ls.autopower(minimum_frequency=fmin, maximum_frequency=fmax, samples_per_peak=5, nyquist_factor=1)
    except Exception as e:
        # fallback: compute power on linear grid
        freq_obj = np.linspace(fmin, fmax, int(np.clip(n_freq_points_per_lc, 200, 5000)))
        ls = LombScargle(t_sec, y_trim, dy_trim)
        power_obj = ls.power(freq_obj, normalization='psd')

    # estimate PSD uncertainty via perturbation if dy present and desired
    if estimate_psd_uncertainty and (dy_trim is not None):
        try:
            # perturb y by ±dy_trim and compute autopower again
            power_plus = LombScargle(t_sec, y_trim + dy_trim, None).autopower(minimum_frequency=fmin, maximum_frequency=fmax, samples_per_peak=5, nyquist_factor=1)[1]
            power_minus = LombScargle(t_sec, y_trim - dy_trim, None).autopower(minimum_frequency=fmin, maximum_frequency=fmax, samples_per_peak=5, nyquist_factor=1)[1]
            sigma_obj = 0.5 * np.abs(power_plus - power_minus)
        except Exception:
            # fallback to NaNs
            sigma_obj = np.full_like(power_obj, np.nan)
    else:
        sigma_obj = np.full_like(power_obj, np.nan)

    # Append arrays (no interpolation, raw)
    all_freqs.append(freq_obj)
    all_pows.append(power_obj)
    all_powerrs.append(sigma_obj)
    contributors.append(os.path.basename(fp))

# Concatenate all points
if len(all_freqs) == 0:
    raise RuntimeError("No PSDs computed for selected files (after trimming).")

all_freqs_concat = np.concatenate(all_freqs)
all_pows_concat  = np.concatenate(all_pows)
all_errs_concat  = np.concatenate(all_powerrs)
# Build contributor list repeated per frequency point
contrib_expanded = []
for fname, freqs_arr in zip(contributors, all_freqs):
    contrib_expanded.extend([fname]*len(freqs_arr))
contrib_expanded = np.array(contrib_expanded)

# Sort by frequency (useful)
order = np.argsort(all_freqs_concat)
all_freqs_concat = all_freqs_concat[order]
all_pows_concat  = all_pows_concat[order]
all_errs_concat  = all_errs_concat[order]
contrib_expanded  = contrib_expanded[order]

# Save concatenated selected PSD points
df_concat = pd.DataFrame({
    "frequency_Hz": all_freqs_concat,
    "power": all_pows_concat,
    "power_err": all_errs_concat,
    "contributor": contrib_expanded
})
f_concat_out = os.path.join(SAVE_FOLDER, "concatenated_selected_psd_points.csv")
df_concat.to_csv(f_concat_out, index=False, float_format="%.12e")
print("Saved concatenated selected PSD points to:", f_concat_out)

In [ ]:
# AGGREGATE BY ROUNDED FREQUENCY
# -----------------------------
# Round frequencies according to freq_round_decimals (adjustable)
rounded_freq = np.round(all_freqs_concat, decimals=freq_round_decimals)

# Group by rounded_freq
unique_freqs, inv_idx = np.unique(rounded_freq, return_inverse=True)
n_unique = unique_freqs.size
print(f"Aggregating into {n_unique} unique rounded frequencies (decimals = {freq_round_decimals}).")

# Prepare aggregation containers
agg_count = np.zeros(n_unique, dtype=int)
agg_values = np.full(n_unique, np.nan, dtype=float)
agg_min = np.full(n_unique, np.nan, dtype=float)
agg_max = np.full(n_unique, np.nan, dtype=float)
agg_minerr = np.full(n_unique, np.nan, dtype=float)  # optionally a propagated error min
agg_maxerr = np.full(n_unique, np.nan, dtype=float)
agg_contributors = [[] for _ in range(n_unique)]

for i,u in enumerate(inv_idx):
    val = all_pows_concat[i]
    errv = all_errs_concat[i]
    agg_contributors[u].append(contrib_expanded[i])
    if np.isnan(agg_values[u]):
        # initialize
        agg_values[u] = val
        agg_min[u] = val
        agg_max[u] = val
        agg_minerr[u] = errv if np.isfinite(errv) else np.nan
        agg_maxerr[u] = errv if np.isfinite(errv) else np.nan
        agg_count[u] = 1
    else:
        # update
        agg_min[u] = min(agg_min[u], val)
        agg_max[u] = max(agg_max[u], val)
        # for aggregate value we will compute later using list comprehension to support median
        agg_count[u] += 1

# compute aggregated metric (median or mean) from the grouped values
# to do this we need to gather values per group (we already have contributors list, but easier to build from inv_idx)
values_per_group = [[] for _ in range(n_unique)]
errs_per_group = [[] for _ in range(n_unique)]
for i,u in enumerate(inv_idx):
    values_per_group[u].append(all_pows_concat[i])
    if np.isfinite(all_errs_concat[i]):
        errs_per_group[u].append(all_errs_concat[i])

agg_value_final = np.full(n_unique, np.nan, dtype=float)
for u in range(n_unique):
    arr = np.array(values_per_group[u], dtype=float)
    if arr.size == 0:
        continue
    if agg_method == "median":
        agg_value_final[u] = np.median(arr)
    else:
        agg_value_final[u] = np.mean(arr)
    # recompute min/max from arr to be safe
    agg_min[u] = np.min(arr)
    agg_max[u] = np.max(arr)
    # propagate error interval: min/max of errors if available
    if len(errs_per_group[u])>0:
        agg_minerr[u] = np.nanmin(errs_per_group[u])
        agg_maxerr[u] = np.nanmax(errs_per_group[u])
    else:
        agg_minerr[u] = np.nan
        agg_maxerr[u] = np.nan

# collect contributors as comma-separated unique list
agg_contributors_str = [",".join(sorted(set(lst))) for lst in agg_contributors]

# build dataframe of aggregated PSD
df_agg = pd.DataFrame({
    "frequency_rounded_Hz": unique_freqs,
    "count": agg_count,
    "agg_value": agg_value_final,
    "min_value": agg_min,
    "max_value": agg_max,
    "min_err": agg_minerr,
    "max_err": agg_maxerr,
    "contributors": agg_contributors_str
})
# sort by frequency ascending
df_agg.sort_values("frequency_rounded_Hz", inplace=True)
agg_out_csv = os.path.join(SAVE_FOLDER, "mean_psd_agg.csv")
df_agg.to_csv(agg_out_csv, index=False, float_format="%.12e")
print("Saved aggregated mean PSD to:", agg_out_csv)

In [ ]:
# -----------------------------
# PLOT raw scatter (selected LCs' PSD points)
# -----------------------------
plt.figure(figsize=(11,5))
plt.scatter(all_freqs_concat, all_pows_concat, s=6, alpha=0.45, rasterized=True)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Frequency [Hz]')
plt.ylabel('Power (PSD)')
plt.title('Raw PSD points (selected & trimmed LCs)')
plt.grid(which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_FOLDER, "raw_selected_psd_scatter.png"), dpi=300)
plt.show()

In [ ]:
# -----------------------------
# PLOT aggregated Mean_PSD with min..max vertical bars (the "error bars" you defined)
# -----------------------------
# mask frequencies with very small count if desired (we keep all here, but you can mask)
mask_plot = df_agg["count"] > 0
freqs_plot = df_agg.loc[mask_plot, "frequency_rounded_Hz"].values
vals_plot = df_agg.loc[mask_plot, "agg_value"].values
min_plot  = df_agg.loc[mask_plot, "min_value"].values
max_plot  = df_agg.loc[mask_plot, "max_value"].values

plt.figure(figsize=(11,6))
plt.plot(freqs_plot, vals_plot, '.-', markersize=4, label=f'agg ({agg_method})')
# draw vertical bars
for x, lo, hi in zip(freqs_plot, min_plot, max_plot):
    plt.vlines(x, ymin=lo, ymax=hi, colors='gray', alpha=0.6, linewidth=1)
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Frequency [Hz]')
plt.ylabel('Aggregated Power (PSD)')
plt.title(f'Aggregated Mean_PSD ({agg_method}) with min..max bars (freq rounding={freq_round_decimals} decimals)')
plt.grid(which='both', alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(SAVE_FOLDER, "mean_psd_aggregated_minmax.png"), dpi=300)
plt.show()


In [ ]:
# -----------------------------
# HISTOGRAM: counts per bin vs frequency grid (bin width hist_bin_width)
# -----------------------------
# Build bin edges between min and max aggregated frequency (or overall freq)
fmin_global = df_agg["frequency_rounded_Hz"].min()
fmax_global = df_agg["frequency_rounded_Hz"].max()
# build linear bins of width hist_bin_width
bin_edges = np.arange(fmin_global, fmax_global + hist_bin_width, hist_bin_width)
if bin_edges.size < 3:
    # fallback to log bins if width is too big
    bin_edges = np.logspace(np.log10(fmin_global), np.log10(fmax_global), 200)
bin_centers = 0.5*(bin_edges[:-1] + bin_edges[1:])
# histogram using concatenated freq points (gives number of PSD points per freq-bin)
counts, _ = np.histogram(all_freqs_concat, bins=bin_edges)

# plot counts vs freq (bar plot); use log y for clarity
plt.figure(figsize=(12,4))
plt.bar(bin_centers, counts, width=(bin_edges[1:] - bin_edges[:-1]), align='center', edgecolor='k', alpha=0.7)
plt.yscale('log')
plt.xscale('log')
plt.xlabel('Frequency [Hz]')
plt.ylabel('Number of PSD points contributing to bin (n_contrib)')
plt.title(f'n_contrib per frequency bin (bin width = {hist_bin_width:.1e} Hz)')
plt.grid(alpha=0.3, which='both')
plt.tight_layout()
hist_out = os.path.join(SAVE_FOLDER, "hist_n_contrib_vs_freq.png")
plt.savefig(hist_out, dpi=300)
plt.show()
# save csv
pd.DataFrame({"bin_center_Hz": bin_centers, "count": counts}).to_csv(os.path.join(SAVE_FOLDER, "n_contrib_per_freq_bin.csv"), index=False, float_format="%.12e")
print("Saved histogram and n_contrib_per_freq_bin.csv")

In [ ]:
# -----------------------------
# Save selected object coordinates (folder names parsed)
# -----------------------------
# We already saved selected_files_list.csv earlier; let's also save unique object folders and coords with high precision
unique_objs = []
coord_rows = []
for base, fp, ra, dec in selected_coords:
    unique_objs.append(base)
    coord_rows.append((base, ra, dec))
unique_objs = sorted(set(unique_objs))
coord_df = pd.DataFrame(coord_rows, columns=["obj_folder", "ra", "dec"])
coord_df.drop_duplicates(subset=["obj_folder"], inplace=True)
coord_df.to_csv(os.path.join(SAVE_FOLDER, "selected_objects_coordinates.csv"), index=False, float_format="%.10f")
# also save a plain txt list of filenames
with open(os.path.join(SAVE_FOLDER, "selected_files_paths.txt"), "w") as fh:
    for fp in selected_files:
        fh.write(fp + "\n")

print("Saved selected object coordinates and file lists in:", SAVE_FOLDER)
print("Done.")